In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


In [ ]:
import sys

package_parent_dir = '/content/gdrive/MyDrive/NLP_assignment_api_client'
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)


In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

In [ ]:
import getpass

API_URL = (__import__("os").getenv("POLI_MILLIONAIRE_API_URL") or input("PoliMillionaire API URL: ").strip())
username = globals().get("username") or input("Poli-Millionaire username: ").strip()
password = globals().get("password") or getpass.getpass("Poli-Millionaire password: ")

In [ ]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

In [ ]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")

In [ ]:
# Choose a competition ID
comp_id = 1

In [ ]:
import json
import re
from urllib.parse import quote, urlencode
from urllib.error import HTTPError
from urllib.request import Request, urlopen
import time

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_USER_AGENT = "PoliMillionaireNLP/1.0 student project"
WIKIPEDIA_REQUEST_DELAY_SECONDS = 0.8
WIKIPEDIA_429_BACKOFF_SECONDS = 4.0
WIKIPEDIA_MAX_RETRIES = 2
MAX_WIKIPEDIA_SEARCH_QUERIES = 2
_LAST_WIKIPEDIA_REQUEST_TIME = 0.0
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that",
    "the", "their", "there", "these", "this", "those", "to", "was", "were",
    "what", "when", "where", "which", "who", "why", "with", "according", "article",
    "considered", "important", "goal", "goals", "main", "primary", "following"
}


def question_to_text(question) -> str:
    """Accept a string, a dict, or a millionaire_client Question object."""
    if hasattr(question, "text"):
        return str(question.text)
    if isinstance(question, dict) and "text" in question:
        return str(question["text"])
    return str(question)


def normalize_wikipedia_text(text: str) -> str:
    """Clean a plain Wikipedia extract enough for later NLP steps."""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text: str) -> list[str]:
    return [token for token in re.findall(r"[a-z0-9]+", str(text).lower()) if len(token) > 1]


def expand_term(token: str) -> set[str]:
    """Tiny synonym/variant helper for common historical wording traps."""
    variants = {token}
    if token == "roman":
        variants.update({"rome", "romans"})
    elif token in {"rome", "romans"}:
        variants.add("roman")
    return variants


def extract_keywords(text: str, limit: int = 10) -> list[str]:
    keywords = []
    seen = set()
    for token in tokenize(text):
        if token in STOPWORDS or token in seen:
            continue
        keywords.append(token)
        seen.add(token)
        if len(keywords) >= limit:
            break
    return keywords


def capital_context_phrases(question_text: str) -> list[str]:
    """Build focused phrases like 'Roman marriage' from capitalized topic words."""
    words = re.findall(r"[A-Za-z][A-Za-z'-]*", question_text)
    phrases = []
    for index, word in enumerate(words):
        if not word[:1].isupper() or word.lower() in STOPWORDS:
            continue
        phrase_words = [word]
        for next_word in words[index + 1:index + 4]:
            if next_word.lower() in STOPWORDS:
                break
            phrase_words.append(next_word)
        if len(phrase_words) > 1:
            phrases.append(" ".join(phrase_words))
    return phrases


def raw_option_text(option) -> str:
    """Read option text without depending on later notebook cells."""
    if hasattr(option, "text"):
        return str(option.text)
    if isinstance(option, dict):
        return str(option.get("text", ""))
    return str(option)


def question_options(question, options=None) -> list:
    """Return answer options from an explicit argument or a Question object."""
    if options is not None:
        return list(options)
    if hasattr(question, "options"):
        return list(question.options)
    if isinstance(question, dict) and "options" in question:
        return list(question["options"])
    return []


def dedupe_queries(queries: list[str]) -> list[str]:
    deduped = []
    seen = set()
    for query in queries:
        normalized = normalize_wikipedia_text(query).lower()
        if normalized and normalized not in seen:
            deduped.append(query)
            seen.add(normalized)
    return deduped


def build_base_wikipedia_search_queries(question) -> list[str]:
    """Create focused question-only Wikipedia queries."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = normalize_wikipedia_text(cleaned)
    keywords = extract_keywords(cleaned, limit=10)

    capital_words = []
    for word in re.findall(r"[A-Za-z][A-Za-z'-]*", cleaned):
        normalized_word = word.strip("'-")
        lowered = normalized_word.lower()
        if normalized_word[:1].isupper() and lowered not in STOPWORDS and len(lowered) > 2:
            capital_words.append(normalized_word)

    queries = []
    if len(capital_words) >= 2:
        queries.append(" ".join(capital_words[:4]))
    queries.extend(capital_context_phrases(cleaned))
    if keywords:
        queries.append(" ".join(keywords[:6]))
    if len(keywords) >= 2:
        queries.append(" ".join(keywords[:2]))
    queries.append(cleaned)
    queries.append(question_text)
    return dedupe_queries(queries)


def build_option_wikipedia_search_queries(question, options=None) -> list[str]:
    """Create one concise search query per option, balanced across all choices."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = normalize_wikipedia_text(cleaned)
    question_keywords = extract_keywords(cleaned, limit=6)
    queries = []
    for option in question_options(question, options):
        option_keywords = extract_keywords(raw_option_text(option), limit=6)
        if option_keywords:
            queries.append(" ".join((question_keywords[:4] + option_keywords[:4])[:8]))
    return dedupe_queries(queries)

def wikipedia_request(params: dict, timeout: float = 6.0, deadline_monotonic=None) -> dict:
    global _LAST_WIKIPEDIA_REQUEST_TIME

    url = f"{WIKIPEDIA_API}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": WIKIPEDIA_USER_AGENT})

    for attempt in range(WIKIPEDIA_MAX_RETRIES + 1):
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            raise TimeoutError("Wikipedia request skipped because the question deadline was reached")

        elapsed_since_last = time.monotonic() - _LAST_WIKIPEDIA_REQUEST_TIME
        sleep_for = WIKIPEDIA_REQUEST_DELAY_SECONDS - elapsed_since_last
        if sleep_for > 0:
            if deadline_monotonic is not None:
                remaining = deadline_monotonic - time.monotonic()
                if remaining <= 0:
                    raise TimeoutError("Wikipedia delay skipped because the question deadline was reached")
                sleep_for = min(sleep_for, remaining)
            time.sleep(sleep_for)

        request_timeout = timeout
        if deadline_monotonic is not None:
            remaining = deadline_monotonic - time.monotonic()
            if remaining <= 0:
                raise TimeoutError("Wikipedia request skipped because the question deadline was reached")
            request_timeout = min(timeout, max(0.25, remaining))

        try:
            with urlopen(request, timeout=request_timeout) as response:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()
                data = json.loads(response.read().decode("utf-8"))
                return data

        except HTTPError as exc:
            if exc.code == 429:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                raise TimeoutError("Wikipedia rate limited; skipping live retry inside timed game")
            if attempt >= WIKIPEDIA_MAX_RETRIES:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                raise

            retry_after = exc.headers.get("Retry-After")
            try:
                wait_seconds = float(retry_after) if retry_after else WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)
            except ValueError:
                wait_seconds = WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)

            if deadline_monotonic is not None:
                remaining = deadline_monotonic - time.monotonic()
                if remaining <= 0 or wait_seconds >= remaining:
                    _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                    raise TimeoutError("Wikipedia rate-limit backoff would exceed the question deadline")
                wait_seconds = min(wait_seconds, remaining)

            print(f"Wikipedia rate limit hit. Waiting {wait_seconds:.1f}s before retry {attempt + 1}/{WIKIPEDIA_MAX_RETRIES}...")
            time.sleep(wait_seconds)
            _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()  # reset AFTER the backoff sleep



def search_wikipedia(query: str, limit: int = 5, timeout: float = 6.0, deadline_monotonic=None) -> list[dict]:
    """Search Wikipedia and return candidate pages for one query string."""
    query = normalize_wikipedia_text(query)
    data = wikipedia_request(
        {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": limit,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
        deadline_monotonic=deadline_monotonic,
    )

    results = data.get("query", {}).get("search", [])
    formatted_results = [
        {
            "title": item.get("title", ""),
            "page_id": item.get("pageid"),
            "snippet": normalize_wikipedia_text(re.sub(r"<[^>]+>", " ", item.get("snippet", ""))),
            "query": query,
            "search_rank": rank,
        }
        for rank, item in enumerate(results, start=1)
    ]
    return formatted_results


def collect_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    """Search capped focused queries and deduplicate candidate pages by title."""
    candidates_by_title = {}
    base_queries = build_base_wikipedia_search_queries(question)
    option_queries = build_option_wikipedia_search_queries(question, options=options)
    if max_search_queries is None:
        max_search_queries = MAX_WIKIPEDIA_SEARCH_QUERIES
    if max_search_queries is not None:
        base_budget = max(1, int(max_search_queries * 0.6))
        option_budget = max(0, max_search_queries - base_budget)
        search_queries = dedupe_queries(base_queries[:base_budget] + option_queries[:option_budget])
        if len(search_queries) < max_search_queries:
            search_queries = dedupe_queries(search_queries + base_queries + option_queries)[:max_search_queries]
    else:
        search_queries = dedupe_queries(base_queries + option_queries)
    for query in search_queries:
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            print("Wikipedia search budget exhausted; using candidates collected so far.")
            break
        try:
            results = search_wikipedia(query, limit=per_query_limit, timeout=timeout, deadline_monotonic=deadline_monotonic)
        except Exception as exc:
            print(f"Wikipedia search skipped for {query!r}: {exc}")
            continue

        for result in results:
            title_key = result["title"].lower()
            if title_key not in candidates_by_title:
                candidates_by_title[title_key] = result
            else:
                candidates_by_title[title_key]["search_rank"] = min(
                    candidates_by_title[title_key]["search_rank"],
                    result["search_rank"],
                )
    return list(candidates_by_title.values())


def core_question_terms(question) -> list[str]:
    """Terms that must anchor retrieval: quoted terms and named entities in the question."""
    question_text = question_to_text(question)
    terms = []
    for phrase in re.findall(r"['\"]([^'\"]{3,80})['\"]", question_text):
        terms.extend(tokenize(phrase))
    for word in re.findall(r"\b[A-Z][A-Za-z0-9'-]{2,}\b", question_text):
        lowered = word.lower().strip("'-")
        if lowered not in STOPWORDS:
            terms.append(lowered)
    return list(dict.fromkeys(terms))[:8]


def candidate_relevance_score(candidate: dict, question, options=None) -> float:
    """Score title/snippet overlap with question keywords; penalize very generic one-word titles."""
    question_text = question_to_text(question)
    keywords = extract_keywords(question_text, limit=10)
    option_keywords = []
    for option in question_options(question, options):
        option_keywords.extend(extract_keywords(raw_option_text(option), limit=5))
    candidate_text = f"{candidate.get('title', '')} {candidate.get('snippet', '')}"
    candidate_terms = set(tokenize(candidate_text))

    matched = 0
    for keyword in keywords:
        if expand_term(keyword) & candidate_terms:
            matched += 1

    overlap = matched / max(1, len(keywords))
    title_terms = tokenize(candidate.get("title", ""))
    core_terms = core_question_terms(question)
    core_overlap = len([term for term in core_terms if expand_term(term) & candidate_terms]) / max(1, len(core_terms)) if core_terms else 1.0
    option_overlap = len(set(option_keywords) & candidate_terms) / max(1, len(set(option_keywords))) if option_keywords else 0.0
    rank_bonus = 1.0 / max(1, candidate.get("search_rank", 1))
    generic_penalty = 0.35 if len(title_terms) == 1 and len(keywords) > 1 else 0.0
    missing_core_penalty = 0.85 if core_terms and core_overlap == 0 else 0.0
    generic_title_penalty = 0.30 if title_terms and core_terms and not (set(title_terms) & set(core_terms)) and len(set(title_terms) & set(keywords)) <= 1 else 0.0

    return (1.6 * overlap) + (0.20 * option_overlap) + (0.80 * core_overlap) + (0.25 * rank_bonus) - generic_penalty - missing_core_penalty - generic_title_penalty


def rank_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    candidates = collect_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries, options=options, deadline_monotonic=deadline_monotonic)
    for candidate in candidates:
        candidate["candidate_score"] = candidate_relevance_score(candidate, question, options=options)
    return sorted(candidates, key=lambda item: item["candidate_score"], reverse=True)


def fetch_wikipedia_extract(title: str, timeout: float = 6.0, deadline_monotonic=None) -> dict:
    """Fetch a Wikipedia page as a plain-text document."""
    data = wikipedia_request(
        {
            "action": "query",
            "prop": "extracts|info",
            "explaintext": 1,
            "exsectionformat": "plain",
            "inprop": "url",
            "titles": title,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
        deadline_monotonic=deadline_monotonic,
    )

    pages = data.get("query", {}).get("pages", {})
    page = next(iter(pages.values()), {}) if pages else {}
    document = {
        "title": page.get("title", title),
        "page_id": page.get("pageid"),
        "url": page.get("fullurl") or f"https://en.wikipedia.org/wiki/{quote(title.replace(' ', '_'))}",
        "text": normalize_wikipedia_text(page.get("extract", "")),
    }
    return document


def get_wikipedia_documents_for_question(question, top_n: int = 5, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    """
    Return the top N related Wikipedia documents for a question.

    This avoids the trap of trusting only Wikipedia's first result for the full question.
    """
    query = question_to_text(question)
    candidates = rank_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries, options=options, deadline_monotonic=deadline_monotonic)
    documents = []

    min_candidate_score = float(globals().get("MIN_WIKIPEDIA_CANDIDATE_SCORE", 0.0))
    for candidate in candidates[:top_n]:
        if candidate.get("candidate_score", 0.0) < min_candidate_score:
            print(f"Wikipedia candidate skipped for low relevance: {candidate.get('title')!r} score={candidate.get('candidate_score', 0.0):.3f}")
            continue
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            print("Wikipedia fetch budget exhausted; using documents fetched so far.")
            break
        try:
            document = fetch_wikipedia_extract(candidate["title"], timeout=timeout, deadline_monotonic=deadline_monotonic)
        except Exception as exc:
            print(f"Wikipedia page skipped for {candidate['title']!r}: {exc}")
            continue

        document["query"] = query
        document["matched_query"] = candidate.get("query")
        document["search_rank"] = candidate.get("search_rank")
        document["candidate_score"] = candidate.get("candidate_score", 0.0)
        document["snippet"] = candidate.get("snippet", "")
        document["search_results"] = candidates
        documents.append(document)

    return documents


In [ ]:

def split_sentences(text: str) -> list[str]:
    """Sentence splitter for clean Wikipedia text."""
    text = normalize_wikipedia_text(text)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [sentence.strip() for sentence in sentences if len(sentence.strip()) >= 40]


def build_rag_chunks(documents: list[dict], sentences_per_chunk: int = 5, overlap: int = 2) -> list[dict]:
    """Split the top-N Wikipedia documents into overlapping evidence chunks."""
    chunks = []
    step = max(1, sentences_per_chunk - overlap)

    for doc_index, doc in enumerate(documents):
        sentences = split_sentences(doc.get("text", ""))
        for start in range(0, len(sentences), step):
            chunk_sentences = sentences[start:start + sentences_per_chunk]
            if not chunk_sentences:
                break
            chunk_text = " ".join(chunk_sentences)
            if len(chunk_text) < 120:
                continue
            chunks.append(
                {
                    "doc_index": doc_index,
                    "chunk_index": len(chunks),
                    "title": doc.get("title", ""),
                    "url": doc.get("url", ""),
                    "text": chunk_text,
                    "document_score": float(doc.get("candidate_score", 0.0)),
                }
            )
            if start + sentences_per_chunk >= len(sentences):
                break

    return chunks


def lexical_similarity(query: str, text: str) -> float:
    """Fallback score if sklearn is unavailable."""
    query_terms = set(extract_keywords(query, limit=20))
    text_terms = set(tokenize(text))
    if not query_terms or not text_terms:
        return 0.0
    overlap = len(query_terms & text_terms) / len(query_terms)
    return overlap

def ensure_pyterrier_started():
    """Import and initialize PyTerrier once for BM25 retrieval."""
    import pyterrier as pt

    started = True
    if hasattr(pt, "started"):
        started = pt.started()
    elif hasattr(pt, "java") and hasattr(pt.java, "started"):
        started = pt.java.started()

    if not started:
        if hasattr(pt, "init"):
            pt.init()
        elif hasattr(pt, "java") and hasattr(pt.java, "init"):
            pt.java.init()

    return pt


def retrieve_rag_chunks_with_tfidf_fallback(question_text: str, chunks: list[dict], top_k: int = 8) -> list[dict]:
    """Fallback retriever used only when PyTerrier is unavailable."""
    chunk_texts = [f"{chunk['title']} {chunk['text']}" for chunk in chunks]

    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity

        vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        matrix = vectorizer.fit_transform([question_text] + chunk_texts)
        similarities = cosine_similarity(matrix[0:1], matrix[1:]).flatten()
        method = "tfidf_cosine_fallback"
    except Exception:
        similarities = [lexical_similarity(question_text, text) for text in chunk_texts]
        method = "lexical_overlap_fallback"

    ranked = []
    for chunk, similarity in zip(chunks, similarities):
        score = float(similarity) + 0.08 * chunk.get("document_score", 0.0)
        enriched = dict(chunk)
        enriched["retrieval_score"] = score
        enriched["retrieval_method"] = method
        ranked.append(enriched)

    ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return ranked[:top_k]


def retrieve_rag_chunks(question, documents: list[dict], top_k: int = 8) -> list[dict]:
    """Retrieve the strongest chunks with PyTerrier BM25 over the Wikipedia chunks."""
    question_text = question_to_text(question)
    bm25_query = " ".join(extract_keywords(question_text, limit=30)) or question_text
    chunks = build_rag_chunks(documents)
    if not chunks:
        return []
    if not globals().get("USE_PYTERRIER_BM25", True):
        return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)

    try:
        import shutil
        import tempfile

        pt = ensure_pyterrier_started()
        index_dir = tempfile.mkdtemp(prefix="pt_rag_chunks_")
        try:
            indexer = pt.IterDictIndexer(index_dir, meta={"docno": 32}, overwrite=True)
            index_ref = indexer.index(
                {
                    "docno": str(index),
                    "text": f"{chunk.get('title', '')} {chunk.get('text', '')}",
                }
                for index, chunk in enumerate(chunks)
            )
            if hasattr(pt, "terrier") and hasattr(pt.terrier, "Retriever"):
                retriever = pt.terrier.Retriever(index_ref, wmodel="BM25", metadata=["docno"])
            else:
                retriever = pt.BatchRetrieve(index_ref, wmodel="BM25", metadata=["docno"])
            results = retriever.search(bm25_query)
        finally:
            shutil.rmtree(index_dir, ignore_errors=True)

        score_by_docno = {
            str(row.docno): float(row.score)
            for row in results.itertuples(index=False)
        }
        max_bm25 = max(score_by_docno.values(), default=0.0)
        if max_bm25 <= 0:
            return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)

        ranked = []
        for index, chunk in enumerate(chunks):
            raw_bm25 = score_by_docno.get(str(index), 0.0)
            normalized_bm25 = raw_bm25 / max_bm25 if max_bm25 else 0.0
            score = normalized_bm25 + 0.08 * chunk.get("document_score", 0.0)
            enriched = dict(chunk)
            enriched["retrieval_score"] = score
            enriched["bm25_score"] = raw_bm25
            enriched["retrieval_method"] = "pyterrier_bm25"
            ranked.append(enriched)

        ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
        return ranked[:top_k]
    except Exception as exc:
        print(f"PyTerrier BM25 retrieval skipped, using fallback retrieval instead: {exc}")
        return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)


def build_rag_context(hits: list[dict], max_chars: int = 4200) -> str:
    """Format retrieved chunks as compact evidence for generation."""
    blocks = []
    used = 0
    for index, hit in enumerate(hits, start=1):
        block = f"[Evidence {index} | {hit['title']}] {hit['text']}"
        if used + len(block) > max_chars:
            block = block[:max(0, max_chars - used)]
        if block.strip():
            blocks.append(block)
            used += len(block)
        if used >= max_chars:
            break
    return "\n\n".join(blocks)


def rank_answer_sentences(question, hits: list[dict], max_sentences: int = 5) -> list[str]:
    """Extract the most relevant evidence sentences for a no-generator answer."""
    question_text = question_to_text(question)
    question_terms = set(tokenize(question_text))
    purpose_terms = {"goal", "purpose", "reason", "important", "considered", "used", "use", "tool", "primarily", "primary", "fundamental", "institution"}
    wants_purpose = bool(question_terms & purpose_terms)
    candidates = []
    seen = set()

    for hit_index, hit in enumerate(hits):
        for sentence_index, sentence in enumerate(split_sentences(hit.get("text", ""))):
            key = sentence.lower()
            if key in seen:
                continue
            seen.add(key)
            sentence_terms = set(tokenize(sentence))
            sentence_for_score = f"{hit.get('title', '')} {sentence}"
            score = lexical_similarity(question_text, sentence_for_score) + 0.15 * hit.get("retrieval_score", 0.0)
            if wants_purpose:
                score += 0.25 * len(sentence_terms & purpose_terms)
            if hit.get("title", "").lower() in sentence.lower():
                score += 0.05
            candidates.append((score, hit_index, sentence_index, sentence))

    candidates.sort(key=lambda item: item[0], reverse=True)
    return [sentence for _, _, _, sentence in candidates[:max_sentences]]


def extractive_rag_answer(question, hits: list[dict], max_sentences: int = 5) -> str:
    """Create a concise explanation paragraph from retrieved evidence sentences."""
    sentences = rank_answer_sentences(question, hits, max_sentences=max_sentences)
    if not sentences:
        return "I could not find enough evidence in the retrieved Wikipedia documents to answer confidently."
    return " ".join(sentences)


QWEN_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
_LOADED_MODELS = {}


def get_huggingface_token():
    """Read a Hugging Face token from Colab Secrets or environment variables, without hard-coding it."""
    import os

    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        token = os.environ.get(name)
        if token:
            return token

    try:
        from google.colab import userdata
        for name in ("hf_token", "HF_TOKEN", "huggingface"):
            token = userdata.get(name)
            if token:
                return token
    except Exception:
        pass

    return None





def generate_local_rag_answer(question, hits: list[dict], model_name: str = QWEN_MODEL_ID, max_new_tokens: int = 180) -> str:
    """Generate an explanatory RAG answer with the local instruct model."""
    import torch

    tokenizer, model = load_instruct_model(model_name)
    question_text = question_to_text(question)
    context = build_rag_context(hits)

    messages = [
        {
            "role": "system",
            "content": "Answer the question using only the retrieved evidence. Be concise and do not invent facts.",
        },
        {
            "role": "user",
            "content": f"Evidence:\n{context}\n\nQuestion: {question_text}\n\nAnswer:",
        },
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer:"

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def answer_question_with_rag(
    question,
    documents: list[dict],
    top_k_chunks: int = 12,
    use_local_generator: bool = True,
    generator_model: str = QWEN_MODEL_ID,
) -> dict:
    """
    Ask the question from RAG using the top-N documents, without using answer options.

    Returns an explanatory answer plus the retrieved evidence chunks.
    """
    hits = retrieve_rag_chunks(question, documents, top_k=top_k_chunks)
    method = "extractive_rag"

    if use_local_generator:
        try:
            answer = generate_local_rag_answer(question, hits, model_name=generator_model)
            method = f"local_rag:{generator_model}"
        except Exception as exc:
            print(f"Local generator skipped, using extractive RAG instead: {exc}")
            answer = extractive_rag_answer(question, hits)
    else:
        answer = extractive_rag_answer(question, hits)

    return {
        "question": question_to_text(question),
        "answer": answer,
        "method": method,
        "evidence_chunks": hits,
    }


In [ ]:
def load_instruct_model(model_name: str = QWEN_MODEL_ID):
    """Load the local instruct model, using 4-bit quantization on CUDA when available."""
    if model_name in _LOADED_MODELS:
        return _LOADED_MODELS[model_name]

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    token = get_huggingface_token()
    tokenizer_kwargs = {"trust_remote_code": True}
    model_kwargs = {"trust_remote_code": True}
    if token:
        tokenizer_kwargs["token"] = token
        model_kwargs["token"] = token

    tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        major, _minor = torch.cuda.get_device_capability()
        compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
        try:
            from transformers import BitsAndBytesConfig
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=compute_dtype,
            )
            model_kwargs.update({
                "quantization_config": quant_config,
                "device_map": {"": 0},
                "low_cpu_mem_usage": True,
            })
        except Exception:
            model_kwargs.update({
                "torch_dtype": compute_dtype,
                "device_map": {"": 0},
                "low_cpu_mem_usage": True,
            })
    else:
        model_kwargs.update({"torch_dtype": torch.float32, "low_cpu_mem_usage": True})

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if not torch.cuda.is_available():
        model.to("cpu")
    model.eval()

    _LOADED_MODELS[model_name] = (tokenizer, model)
    return tokenizer, model


In [ ]:
SYSTEM_PROMPT = """
You are an expert multiple choice quiz solver.

Carefully analyze the question.

Return ONLY the single best answer option.

Do not explain your reasoning.
Do not output extra text.
"""

def build_mcq_prompt(question):

    choices = []

    for i, opt in enumerate(question.options):
        letter = chr(ord("A") + i)
        choices.append(f"{letter}. {option_text(opt)}")

    joined = "\n".join(choices)

    return f"""
Question:
{question.text}

Options:
{joined}

Reply with ONLY one letter: A, B, C, or D.
"""

In [ ]:
LETTERS = "ABCD"


def option_text(option) -> str:
    return option.text if hasattr(option, "text") else option["text"]


def option_id(option) -> int:
    return option.id if hasattr(option, "id") else option["id"]


def parse_option_choice(text: str, option_count: int = 4):
    """Parse A-D or 0-3 from the model output."""
    cleaned = str(text).strip().upper()

    letter_match = re.search(r"\b([A-D])\b", cleaned)
    if letter_match:
        index = LETTERS.index(letter_match.group(1))
        return index if index < option_count else None

    digit_match = re.search(r"\b([0-3])\b", cleaned)
    if digit_match:
        index = int(digit_match.group(1))
        return index if index < option_count else None

    return None


def parse_option_choice_with_text(text: str, options):
    """Parse the chosen option, preferring an exact option-text mention over a possibly wrong letter."""
    normalized_output = normalize_match_text(text)
    text_matches = []
    for index, option in enumerate(options):
        normalized_option = normalize_match_text(option_text(option))
        if normalized_option and normalized_option in normalized_output:
            text_matches.append(index)
    if len(text_matches) == 1:
        return text_matches[0]
    return parse_option_choice(text, option_count=len(options))

def choose_option_direct_shuffled_logits(question, options, model_name: str = QWEN_MODEL_ID) -> dict:
    """Average next-letter logit scores across shuffled option orders."""
    import hashlib
    import random
    import torch
    import torch.nn.functional as F

    vote_count = max(3, int(globals().get("DIRECT_MODEL_VOTES", 3)))
    options = list(options)

    question_text = question_to_text(question)
    tokenizer, model = load_instruct_model(model_name)
    device = next(model.parameters()).device

    score_lists = {index: [] for index in range(len(options))}
    vote_details = []

    for vote_number in range(vote_count):
        order = list(range(len(options)))

        if vote_number > 0:
            seed = int(
                hashlib.sha256(
                    f"{question_text}|logits|{vote_number}".encode("utf-8")
                ).hexdigest()[:12],
                16,
            )
            rng = random.Random(seed)
            rng.shuffle(order)

            if order == list(range(len(options))) and len(order) > 1:
                order = order[1:] + order[:1]

        option_lines = "\n".join(
            f"{LETTERS[display_index]}. {option_text(options[original_index])}"
            for display_index, original_index in enumerate(order)
        )

        user_prompt = f"""
Question:
{question_text}

Options:
{option_lines}

Reply with ONLY one letter: A, B, C, or D.
"""

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template"):
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1536,
        ).to(device)

        with torch.inference_mode():
            outputs = model(**inputs)
            log_probs = F.log_softmax(outputs.logits[0, -1], dim=-1)

        display_scores = []

        for display_index, original_index in enumerate(order):
            letter = LETTERS[display_index]
            variant_scores = []

            for variant in (letter, f" {letter}", f"{letter}.", f" {letter}."):
                token_ids = tokenizer.encode(variant, add_special_tokens=False)
                if token_ids:
                    variant_scores.append(float(log_probs[token_ids[0]].detach().cpu()))

            best_score = max(variant_scores) if variant_scores else float("-inf")

            score_lists[original_index].append(best_score)
            display_scores.append({
                "display_letter": letter,
                "answer_index": original_index,
                "logprob": best_score,
            })

        display_scores.sort(key=lambda item: item["logprob"], reverse=True)

        vote_details.append({
            "vote_number": vote_number + 1,
            "display_order": order,
            "ranked": display_scores,
        })

    averaged_scores = []

    for index, scores in score_lists.items():
        averaged_scores.append({
            "answer_index": index,
            "letter": LETTERS[index],
            "avg_logprob": sum(scores) / max(1, len(scores)),
            "logprobs": scores,
        })

    ranked = sorted(
        averaged_scores,
        key=lambda item: item["avg_logprob"],
        reverse=True,
    )

    selected_index = ranked[0]["answer_index"]
    margin = ranked[0]["avg_logprob"] - ranked[1]["avg_logprob"] if len(ranked) > 1 else 0.0
    selected_option = options[selected_index]

    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_index,
        "letter": LETTERS[selected_index],
        "model_output": "shuffled_logit_scores:" + ", ".join(
            f"{item['letter']}={item['avg_logprob']:.3f}" for item in ranked
        ),
        "selection_source": f"direct_model_shuffled_logit_scoring_{vote_count}",
        "direct_logit_scores": ranked,
        "direct_logit_margin": margin,
        "direct_votes": vote_details,
        "option_scores": [],
    }


def choose_option_direct_voted(question, options, model_name: str = QWEN_MODEL_ID) -> dict:
    """Fast direct answer with strict MCQ voting."""
    vote_count = int(globals().get("DIRECT_MODEL_VOTES", 1))
    vote_count = max(1, vote_count)

    tokenizer, model = load_instruct_model(model_name)
    user_prompt = build_mcq_prompt(question)

    import torch

    device = next(model.parameters()).device
    votes = []

    for _ in range(vote_count):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template"):
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1536,
        ).to(device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=2,
                do_sample=False,
                temperature=0.0,
                top_p=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        model_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        selected_index = parse_option_choice_with_text(model_output, options)

        if selected_index is not None:
            votes.append({
                "answer_index": selected_index,
                "letter": LETTERS[selected_index],
                "model_output": model_output,
            })

    if not votes:
        raise ValueError("Could not parse any direct model vote")

    counts = {}
    first_seen = {}

    for order, vote in enumerate(votes):
        index = vote["answer_index"]
        counts[index] = counts.get(index, 0) + 1
        first_seen.setdefault(index, order)

    selected_index = sorted(
        counts,
        key=lambda index: (-counts[index], first_seen[index]),
    )[0]

    selected_option = options[selected_index]

    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_index,
        "letter": LETTERS[selected_index],
        "model_output": " | ".join(vote["model_output"] for vote in votes),
        "selection_source": f"direct_model_voted_{len(votes)}",
        "direct_votes": votes,
        "option_scores": [],
    }

def normalize_match_text(text: str) -> str:
    """Normalize text for cheap exact/near-exact option matching."""
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", str(text).lower())).strip()


def score_options_against_evidence(question, options, hits: list[dict]) -> list[dict]:
    """Score each option directly against retrieved evidence using fast lexical/exact matching."""
    question_text = question_to_text(question)
    evidence_texts = [f"{hit.get('title', '')} {hit.get('text', '')}" for hit in hits if hit.get("text")]
    evidence_blob = " ".join(evidence_texts)
    normalized_evidence = normalize_match_text(evidence_blob)
    question_terms = set(extract_keywords(question_text, limit=24))
    evidence_sentences = split_sentences(evidence_blob) if evidence_blob else []
    try:
        core_terms = core_question_terms(question)
    except NameError:
        core_terms = []
    evidence_terms = set(tokenize(evidence_blob)) if evidence_blob else set()
    evidence_core_overlap = len([term for term in core_terms if expand_term(term) & evidence_terms]) / max(1, len(core_terms)) if core_terms else 1.0
    topic_relevance_multiplier = 1.0 if evidence_core_overlap > 0 else 0.35
    scored = []

    lexical_scores = []
    for option in options:
        option_query = f"{question_text} {option_text(option)}"
        if evidence_texts:
            lexical_scores.append(max(lexical_similarity(option_query, evidence_text) for evidence_text in evidence_texts))
        else:
            lexical_scores.append(0.0)

    for index, option in enumerate(options):
        lexical_score = float(lexical_scores[index])
        normalized_option = normalize_match_text(option_text(option))
        option_terms = [term for term in normalized_option.split() if len(term) > 2]
        exact_match = bool(normalized_option and normalized_option in normalized_evidence)
        support_sentence_score = 0.0
        support_sentence = ""
        for sentence in evidence_sentences:
            normalized_sentence = normalize_match_text(sentence)
            if not normalized_option or normalized_option not in normalized_sentence:
                continue
            sentence_terms = set(tokenize(sentence))
            overlap = len(question_terms & sentence_terms) / max(1, len(question_terms))
            score = 0.75 + overlap
            if "only once" in normalized_sentence or "rarely" in normalized_sentence:
                score -= 0.75
            if score > support_sentence_score:
                support_sentence_score = score
                support_sentence = sentence[:260]
        term_coverage = len([term for term in option_terms if term in normalized_evidence]) / max(1, len(option_terms))
        exact_boost = float(globals().get("EXACT_OPTION_MATCH_BOOST", 1.8)) if exact_match else 0.0
        coverage_boost = float(globals().get("OPTION_TERM_COVERAGE_WEIGHT", 0.35)) * term_coverage
        combined_score = (lexical_score + exact_boost + coverage_boost + support_sentence_score) * topic_relevance_multiplier
        scored.append(
            {
                "answer_id": option_id(option),
                "answer_text": option_text(option),
                "answer_index": index,
                "letter": LETTERS[index],
                "lexical_score": lexical_score,
                "exact_match": exact_match,
                "term_coverage": term_coverage,
                "support_sentence_score": support_sentence_score,
                "support_sentence": support_sentence,
                "evidence_core_overlap": evidence_core_overlap,
                "combined_score": combined_score,
            }
        )

    scored.sort(key=lambda item: item["combined_score"], reverse=True)
    return scored


In [ ]:
def is_numeric_question(question) -> bool:
    q = question.text.lower()
    option_texts = " ".join(option_text(opt) for opt in question.options)

    numeric_words = [
        "population", "year", "date", "century", "how many",
        "number", "estimated", "amount", "percentage", "million",
        "billion", "km", "meters", "age"
    ]

    has_digit_option = bool(re.search(r"\d", option_texts))
    has_numeric_word = any(word in q for word in numeric_words)

    return has_digit_option or has_numeric_word

In [ ]:
def answer_one_question_with_pipeline(question, game=None) -> dict:
    start = time.monotonic()
    seconds_left_start = seconds_available(game) if game is not None else None
    direct_option_match = None
    direct_model_seconds = 0.0

    if seconds_left_start is not None and seconds_left_start < MIN_SECONDS_FOR_ANY_MODEL:
        selected = fallback_option(question)
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [],
            "rag_answer": "Skipped pipeline because too little time remained.",
            "rag_method": "hard_time_guard",
            "evidence_chunks": [],
            "option_match": {
                "answer_id": option_id(selected),
                "answer_text": option_text(selected),
                "answer_index": 0,
                "letter": "A",
                "model_output": "hard_time_guard",
                "selection_source": "hard_time_guard",
                "option_scores": [],
            },
            "elapsed_seconds": 0.0,
            "timings": {},
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_left_start,
        }

    if USE_DIRECT_MODEL_FIRST and (
        seconds_left_start is None
        or seconds_left_start >= QUESTION_TIME_BUFFER + MIN_SECONDS_FOR_DIRECT_MODEL
    ):
        direct_start = time.monotonic()
        try:
            if USE_DIRECT_LOGIT_SCORING:
                direct_option_match = choose_option_direct_shuffled_logits(
                    question,
                    question.options,
                    model_name=GAME_MODEL_ID,
                )
                if direct_option_match.get("direct_logit_margin", 0.0) < DIRECT_LOGIT_CONFIDENCE_MARGIN:
                    print(
                        f"Low direct logit margin "
                        f"({direct_option_match.get('direct_logit_margin', 0.0):.3f}); trying prompt voting."
                    )
                    voted_match = choose_option_direct_voted(
                        question,
                        question.options,
                        model_name=GAME_MODEL_ID,
                    )
                    voted_match["logit_match"] = direct_option_match
                    direct_option_match = voted_match
            else:
                direct_option_match = choose_option_direct_voted(
                    question,
                    question.options,
                    model_name=GAME_MODEL_ID,
                )

            direct_end = time.monotonic()
            direct_model_seconds = direct_end - direct_start

        except Exception as exc:
            print(f"Direct model answer failed; falling back to Wikipedia: {exc}")


    if direct_option_match is not None:
        current_seconds_left = seconds_available(game) if game is not None else seconds_left_start
        direct_margin = direct_logit_margin(direct_option_match)
        use_tool_router = globals().get("USE_AGENTIC_TOOL_ROUTER", True)
        skip_margin = float(globals().get("DIRECT_TOOL_SKIP_MARGIN", DIRECT_LOGIT_CONFIDENCE_MARGIN))
        verify_direct = globals().get("VERIFY_DIRECT_WITH_WIKIPEDIA", True)
        has_tool_time = current_seconds_left is None or current_seconds_left >= QUESTION_TIME_BUFFER + MIN_SECONDS_TO_VERIFY_DIRECT
        high_confidence_direct = use_tool_router and direct_margin >= skip_margin
        should_skip_tools = high_confidence_direct or not verify_direct or not has_tool_time

        if should_skip_tools:
            if high_confidence_direct:
                router_reason = f"high_direct_margin:{direct_margin:.3f}>={skip_margin:.3f}"
            elif not verify_direct:
                router_reason = "verification_disabled"
            else:
                router_reason = "not_enough_time_for_tool_call"
            direct_option_match["selection_source"] = "agentic_router_direct_answer"
            direct_option_match["tool_router_reason"] = router_reason
            direct_option_match["direct_logit_margin"] = direct_margin
            after_match = time.monotonic()
            return {
                "question": question.text,
                "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
                "documents": [],
                "rag_answer": f"Tool router skipped live Wikipedia ({router_reason}); used direct {GAME_MODEL_ID} answer.",
                "rag_method": "agentic_router_direct_answer",
                "evidence_chunks": [],
                "option_match": direct_option_match,
                "elapsed_seconds": after_match - start,
                "timings": {
                    "direct_model_seconds": direct_model_seconds,
                    "wikipedia_seconds": 0.0,
                    "rag_generation_seconds": 0.0,
                    "option_matching_seconds": after_match - start - direct_model_seconds,
                    "pre_submit_pipeline_seconds": after_match - start,
                },
                "seconds_left_start": seconds_left_start,
                "seconds_left_end": seconds_available(game) if game is not None else None,
            }

        print(f"Tool router: direct answer was low confidence (margin={direct_margin:.3f}); calling Wikipedia tool path.")

    current_seconds_left = seconds_available(game) if game is not None else seconds_left_start
    if current_seconds_left is not None:
        final_model_reserve = MIN_SECONDS_FOR_FINAL_MODEL
        retrieval_budget = max(0.0, current_seconds_left - QUESTION_TIME_BUFFER - final_model_reserve)
        retrieval_budget = min(WIKIPEDIA_TIME_BUDGET, retrieval_budget)
    else:
        retrieval_budget = WIKIPEDIA_TIME_BUDGET

    retrieval_deadline = time.monotonic() + max(0.0, retrieval_budget)

    if USE_LIVE_WIKIPEDIA:
        docs = get_wikipedia_documents_for_question(
            question,
            top_n=TOP_N_DOCS,
            per_query_limit=PER_QUERY_LIMIT,
            timeout=WIKIPEDIA_TIMEOUT,
            max_search_queries=MAX_SEARCH_QUERIES,
            options=question.options,
            deadline_monotonic=retrieval_deadline,
        )
    else:
        docs = []

    after_wikipedia = time.monotonic()
    seconds_left_after_wiki = seconds_available(game) if game is not None else None

    if seconds_left_after_wiki is not None and seconds_left_after_wiki <= QUESTION_TIME_BUFFER:
        if direct_option_match is not None:
            option_match = direct_option_match
            option_match["selection_source"] = "direct_model_after_wikipedia_time_guard"
        else:
            option_match = score_fallback_option_match(
                question,
                [],
                "submit_time_guard_after_wikipedia",
            )
        after_match = time.monotonic()
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [
                {
                    "title": doc.get("title"),
                    "url": doc.get("url"),
                    "candidate_score": doc.get("candidate_score"),
                    "matched_query": doc.get("matched_query"),
                }
                for doc in docs
            ],
            "rag_answer": "Skipped chunk retrieval because submit buffer was reached.",
            "rag_method": "submit_time_guard_after_wikipedia",
            "evidence_chunks": [],
            "option_match": option_match,
            "elapsed_seconds": after_match - start,
            "timings": {
                "wikipedia_seconds": after_wikipedia - start,
                "pre_submit_pipeline_seconds": after_match - start,
            },
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_available(game) if game is not None else None,
        }

    if not docs:
        if direct_option_match is not None:
            option_match = direct_option_match
            option_match["selection_source"] = "direct_model_no_wikipedia_docs"
        else:
            option_match = choose_option_direct_voted(
                question,
                question.options,
                model_name=GAME_MODEL_ID,
            )
            option_match["selection_source"] = "direct_model_no_wikipedia_docs_retry"

        after_match = time.monotonic()
        if direct_option_match is not None:
            evidence_scores = option_match.get("option_scores", [])
            evidence_is_strong = False

            if evidence_scores and len(evidence_scores) >= 2:
                top = float(evidence_scores[0].get("combined_score", 0.0))
                second = float(evidence_scores[1].get("combined_score", 0.0))
                evidence_is_strong = (
                top >= MIN_EVIDENCE_SCORE_TO_TRUST
                and (top - second) >= EVIDENCE_CONFIDENCE_MARGIN
                )

            if not evidence_is_strong:
                option_match = direct_option_match
                option_match["selection_source"] = "direct_model_preferred_over_weak_rag"
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [],
            "rag_answer": f"No Wikipedia documents; used direct {GAME_MODEL_ID} answer.",
            "rag_method": "no_docs_direct_model",
            "evidence_chunks": [],
            "option_match": option_match,
            "elapsed_seconds": after_match - start,
            "timings": {
                "direct_model_seconds": direct_model_seconds,
                "wikipedia_seconds": after_wikipedia - start - direct_model_seconds,
                "pre_submit_pipeline_seconds": after_match - start,
            },
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_available(game) if game is not None else None,
     }

    seconds_left_before_rag = seconds_available(game) if game is not None else None
    allow_rag_generator = GENERATE_RAG_DRAFT and (
        seconds_left_before_rag is None
        or seconds_left_before_rag >= QUESTION_TIME_BUFFER + MIN_SECONDS_FOR_FINAL_MODEL
    )

    rag_result = answer_question_with_rag(
        question,
        docs,
        top_k_chunks=TOP_K_CHUNKS,
        use_local_generator=allow_rag_generator,
        generator_model=GAME_MODEL_ID,
    )

    after_rag = time.monotonic()
    seconds_left_after_rag = seconds_available(game) if game is not None else None

    evidence_hits_for_scoring = [{"title": "RAG answer", "text": rag_result.get("answer", "")}] + rag_result["evidence_chunks"]

    evidence_match = score_fallback_option_match(
        question,
        evidence_hits_for_scoring,
        "wikipedia_evidence_score",
    )

    evidence_scores = evidence_match.get("option_scores", [])
    evidence_is_strong = False

    if evidence_scores and len(evidence_scores) >= 2:
        top = float(evidence_scores[0].get("combined_score", 0.0))
        second = float(evidence_scores[1].get("combined_score", 0.0))
        evidence_is_strong = (
            top >= MIN_EVIDENCE_SCORE_TO_TRUST
            and (top - second) >= EVIDENCE_CONFIDENCE_MARGIN
        )


    if direct_option_match is not None and is_numeric_question(question):
        option_match = direct_option_match
        option_match["selection_source"] = "direct_model_numeric_question"

    elif direct_option_match is not None:
        option_match = reconcile_direct_and_evidence(direct_option_match, evidence_match)

    elif evidence_is_strong:
        option_match = evidence_match
        option_match["selection_source"] = "strong_wikipedia_evidence"

    else:
        option_match = evidence_match
        option_match["selection_source"] = "fallback_wikipedia_evidence"

    after_match = time.monotonic()
    elapsed = after_match - start

    return {
        "question": question.text,
        "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
        "documents": [
            {
                "title": doc.get("title"),
                "url": doc.get("url"),
                "candidate_score": doc.get("candidate_score"),
                "matched_query": doc.get("matched_query"),
            }
            for doc in docs
        ],
        "rag_answer": rag_result["answer"],
        "rag_method": rag_result["method"],
        "evidence_chunks": [
            {
                "title": hit.get("title"),
                "retrieval_score": hit.get("retrieval_score"),
                "bm25_score": hit.get("bm25_score"),
                "retrieval_method": hit.get("retrieval_method"),
                "text": hit.get("text", "")[:900],
            }
            for hit in rag_result["evidence_chunks"][:5]
        ],
        "option_match": option_match,
        "elapsed_seconds": elapsed,
        "timings": {
            "direct_model_seconds": direct_model_seconds,
            "wikipedia_seconds": after_wikipedia - start - direct_model_seconds,
            "rag_generation_seconds": after_rag - after_wikipedia,
            "option_matching_seconds": after_match - after_rag,
            "pre_submit_pipeline_seconds": elapsed,
        },
        "seconds_left_start": seconds_left_start,
        "seconds_left_end": seconds_available(game) if game is not None else None,
    }

In [ ]:
# =========================
# GAME CONFIG AND HELPERS: direct model + optional Wikipedia RAG verification
# =========================
# Run the setup, login, retrieval, and model helper cells above first.

comp_id = 1  # Set your competition ID here.

# ---- Hyperparameters ----
RUN_ACTUAL_GAME = True
COMPETITION_ID = comp_id
MAX_QUESTIONS = None          # Use 1 or 2 for a small test; None = play until game over.
SUBMIT_ANSWERS = True         # True = send answers to API. False = dry run, no submission.

MAX_SEARCH_QUERIES = 2        # Free Colab/timed game: keep live Wikipedia small to avoid 429s.
PER_QUERY_LIMIT = 2           # Enough fallback candidates without burning the whole timer.
TOP_N_DOCS = 3                # Smaller evidence set for a 30-second question window.



WIKIPEDIA_TIMEOUT = 2.0       # Seconds per Wikipedia API request.
WIKIPEDIA_DELAY_SECONDS = 1.0 # Respect Wikipedia's rate limit.
WIKIPEDIA_BACKOFF_SECONDS = 0.5
WIKIPEDIA_RETRIES = 0
MIN_WIKIPEDIA_CANDIDATE_SCORE = 0.25 # Fetch more plausible pages; evidence scorer filters them later.
TOP_K_CHUNKS = 12          # Keep the final evidence set small for the 30-second timer.
USE_LIVE_WIKIPEDIA = True    # Call Wikipedia during the timed game.
USE_PYTERRIER_BM25 = True      # Timed game: TF-IDF fallback is much faster for small per-question chunks.
USE_DIRECT_MODEL_FIRST = True   # Cheap first opinion; retrieved evidence can verify or correct it.
USE_DIRECT_LOGIT_SCORING = True    # Use stable direct next-token logit scoring for no-evidence fallback.
DIRECT_LOGIT_CONFIDENCE_MARGIN = 1.0 # Low margin triggers vote/Wikipedia fallback.
USE_AGENTIC_TOOL_ROUTER = True # Direct answer first; call Wikipedia only when confidence/time says it is worth it.
DIRECT_TOOL_SKIP_MARGIN = 1.25 # If direct logit margin reaches this, skip tool calls and submit.
VERIFY_DIRECT_WITH_WIKIPEDIA = True # Router may use Wikipedia for low-confidence direct answers.
DIRECT_MODEL_VOTES = 3        # Used only by optional generated-vote helpers, not the default fallback.
GENERATE_RAG_DRAFT = False  # Timed game: avoid slow explanatory generation before choosing an option.
QUESTION_TIME_BUFFER = 6.0    # Submit before this many seconds remain.
MIN_SECONDS_TO_ATTEMPT = 4.0  # If less time remains, submit fallback option 0.
WIKIPEDIA_TIME_BUDGET = 5.0   # Hard cap for Wikipedia search + page fetch.
MIN_SECONDS_FOR_FINAL_MODEL = 8.0 # Require this much extra time, after the submit buffer, before final model.
MIN_SECONDS_FOR_ANY_MODEL = 3.0   # Below this, submit fallback immediately.
MIN_SECONDS_FOR_DIRECT_MODEL = 4.0
MIN_SECONDS_TO_VERIFY_DIRECT = 10.0
EVIDENCE_OVERRIDE_MARGIN = 0.20
EVIDENCE_CONFIDENCE_MARGIN = 0.12
MIN_EVIDENCE_SCORE_TO_TRUST = 0.45
MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE = 2.50
EXACT_OPTION_MATCH_BOOST = 1.8
OPTION_TERM_COVERAGE_WEIGHT = 0.35

DELAY_SUBMIT_FOR_WIKI_COOLDOWN = False # Never wait on purpose inside a 30-second question.
TARGET_SUBMIT_ELAPSED_SECONDS = 22.0
MIN_SECONDS_LEFT_AT_SUBMIT = 5.0      # Safety margin for network/server latency.
MAX_SUBMIT_WAIT_SECONDS = 0

GAME_MODEL_ID = QWEN_MODEL_ID
MANUAL_HF_TOKEN = ""          # Prefer Colab Secrets HF_TOKEN. Temporary manual token goes here if needed.
PRELOAD_MODEL = True
SAVE_RUN_LOG = True
RUN_LOG_DIR = "/content/gdrive/MyDrive/NLP_assignment/test3_rag_game_runs"
VERBOSE = True

# ---- End hyperparameters ----

import getpass
import importlib.util
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def ensure_runtime_packages():
    required_packages = [
        ("transformers", "transformers"),
        ("accelerate", "accelerate"),
        ("bitsandbytes", "bitsandbytes"),
        ("scikit-learn", "sklearn"),
    ]
    if globals().get("USE_PYTERRIER_BM25", False):
        required_packages.append(("python-terrier", "pyterrier"))
    missing = [package for package, module in required_packages if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


def ensure_hf_token_for_game():
    token = get_huggingface_token() or MANUAL_HF_TOKEN.strip()
    if not token:
        token = getpass.getpass("Hugging Face token (input hidden): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
    if not get_huggingface_token():
        raise RuntimeError("No Hugging Face token found. Add HF_TOKEN in Colab Secrets, set MANUAL_HF_TOKEN, or enter it when prompted.")


def preload_model_for_game():
    ensure_runtime_packages()
    ensure_hf_token_for_game()
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
    globals()["USE_PYTERRIER_BM25"] = USE_PYTERRIER_BM25
    print(f"Preloading {GAME_MODEL_ID} before starting timed game...")
    start = time.time()
    try:
        tokenizer, model = load_instruct_model(GAME_MODEL_ID)
    except Exception as exc:
        raise RuntimeError(
            "Could not load the local Qwen model. Restart the Colab runtime, run only the setup cells, "
            "and use a GPU runtime if available. The timed game was not started."
        ) from exc
    active_model_name = next((name for name, loaded in _LOADED_MODELS.items() if loaded == (tokenizer, model)), GAME_MODEL_ID)
    globals()["GAME_MODEL_ID"] = active_model_name
    model_device = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype
    print(f"Loaded model: {GAME_MODEL_ID} | device={model_device} | dtype={model_dtype}")

    import torch

    warmup_messages = [
        {"role": "system", "content": "Answer briefly."},
        {"role": "user", "content": "Say ready."},
    ]
    warmup_prompt = tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
    device = next(model.parameters()).device
    inputs = tokenizer(warmup_prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        _ = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    print(f"Direct model ready in {time.time() - start:.1f}s. Starting game only after this point.")


def seconds_available(game) -> float:
    remaining = game.time_remaining
    if remaining is None:
        return 30.0
    return max(0.0, float(remaining))


def fallback_option(question):
    return question.options[0]


def score_fallback_option_match(question, hits: list[dict], reason: str) -> dict:
    """Choose the highest evidence-score option without another model call."""
    option_scores = score_options_against_evidence(question, question.options, hits)
    best_score = option_scores[0] if option_scores else {"answer_index": 0}
    selected_option = question.options[best_score["answer_index"]]
    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": best_score["answer_index"],
        "letter": LETTERS[best_score["answer_index"]],
        "model_output": reason,
        "selection_source": "timed_option_evidence_score_fallback",
        "selected_option_score": best_score,
        "option_scores": option_scores,
    }


def reconcile_direct_and_evidence(direct_match: dict, evidence_match: dict) -> dict:
    """Keep direct model unless evidence strongly supports another option."""
    if not direct_match:
        return evidence_match
    if not evidence_match or not evidence_match.get("option_scores"):
        direct = dict(direct_match)
        direct["selection_source"] = "direct_model_no_evidence"
        return direct

    scores = evidence_match["option_scores"]
    top = scores[0]
    direct_score = next((item for item in scores if item["answer_index"] == direct_match["answer_index"]), None)
    direct_value = float(direct_score.get("combined_score", 0.0)) if direct_score else 0.0
    top_value = float(top.get("combined_score", 0.0))
    second_value = float(scores[1].get("combined_score", 0.0)) if len(scores) > 1 else 0.0
    evidence_margin = top_value - second_value
    margin = top_value - direct_value
    direct_margin = direct_logit_margin(direct_match)
    direct_is_high_confidence = direct_margin >= DIRECT_LOGIT_CONFIDENCE_MARGIN
    direct_is_safe_with_weak_evidence = direct_margin >= globals().get("MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE", 1.20)
    top_exact_with_margin = bool(top.get("exact_match")) and evidence_margin >= EVIDENCE_CONFIDENCE_MARGIN
    top_is_trustworthy = top_exact_with_margin or (
        top_value >= MIN_EVIDENCE_SCORE_TO_TRUST and evidence_margin >= EVIDENCE_OVERRIDE_MARGIN
    )
    should_override = top["answer_index"] != direct_match["answer_index"] and top_is_trustworthy and margin >= EVIDENCE_OVERRIDE_MARGIN

    if should_override:
        chosen = dict(evidence_match)
        chosen["selection_source"] = "wikipedia_evidence_overrode_direct_model"
        chosen["direct_model_match"] = direct_match
        chosen["evidence_override_margin"] = margin
        chosen["evidence_score_margin"] = evidence_margin
        return chosen

    chosen = dict(direct_match)
    if top["answer_index"] == direct_match["answer_index"] and top_is_trustworthy:
        chosen["selection_source"] = "direct_model_confirmed_by_wikipedia_evidence"
    elif direct_is_safe_with_weak_evidence:
        chosen["selection_source"] = "direct_model_high_confidence_weak_evidence"
    elif direct_is_high_confidence:
        chosen["selection_source"] = "direct_model_acceptable_margin_weak_evidence"
    else:
        chosen = dict(evidence_match)
        chosen["selection_source"] = "weak_direct_model_deferred_to_evidence_score"
        chosen["direct_model_match"] = direct_match
        chosen["evidence_override_margin"] = margin
        chosen["evidence_score_margin"] = evidence_margin
        return chosen
    chosen["option_scores"] = scores
    chosen["selected_option_score"] = direct_score
    chosen["evidence_top_option"] = top
    chosen["evidence_override_margin"] = margin
    chosen["evidence_score_margin"] = evidence_margin
    chosen["direct_logit_margin"] = direct_margin
    return chosen


def direct_logit_margin(direct_match: dict) -> float:
    """Return the direct logit margin, including when a generated vote wrapped it."""
    if not direct_match:
        return 0.0
    if "direct_logit_margin" in direct_match:
        return float(direct_match.get("direct_logit_margin", 0.0))
    logit_match = direct_match.get("logit_match") or {}
    return float(logit_match.get("direct_logit_margin", 0.0))


def wait_before_submit_for_cooldown(game) -> float:
    """Wait after computing the answer so Wikipedia gets cooldown time before next question."""
    if not DELAY_SUBMIT_FOR_WIKI_COOLDOWN:
        return 0.0

    current_remaining = seconds_available(game)
    target_remaining = max(MIN_SECONDS_LEFT_AT_SUBMIT, 30.0 - TARGET_SUBMIT_ELAPSED_SECONDS)
    wait_seconds = current_remaining - target_remaining
    wait_seconds = min(MAX_SUBMIT_WAIT_SECONDS, max(0.0, wait_seconds))

    if wait_seconds > 0:
        print(f"Answer ready. Waiting {wait_seconds:.1f}s before submit to give Wikipedia API cooldown time.")
        time.sleep(wait_seconds)

    return wait_seconds



def play_actual_rag_game():
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
    globals()["USE_PYTERRIER_BM25"] = USE_PYTERRIER_BM25

    if GAME_MODEL_ID not in _LOADED_MODELS:
        raise RuntimeError("Run the model preload cell before starting the actual game.")

    # Keep the Wikipedia cooldown state across setup/game questions so live requests do not trip 429s.

    if not RUN_ACTUAL_GAME:
        print("RUN_ACTUAL_GAME is False. Set it to True to start a real timed game.")
        return None, None

    game = client.game.start(competition_id=COMPETITION_ID)
    run_log = {
        "session_id": game.session_id,
        "competition_id": COMPETITION_ID,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "top_n_docs": TOP_N_DOCS,
            "per_query_limit": PER_QUERY_LIMIT,
            "max_search_queries": MAX_SEARCH_QUERIES,
            "wikipedia_timeout": WIKIPEDIA_TIMEOUT,
            "wikipedia_delay_seconds": WIKIPEDIA_DELAY_SECONDS,
            "wikipedia_backoff_seconds": WIKIPEDIA_BACKOFF_SECONDS,
            "wikipedia_retries": WIKIPEDIA_RETRIES,
            "min_wikipedia_candidate_score": MIN_WIKIPEDIA_CANDIDATE_SCORE,
            "use_live_wikipedia": USE_LIVE_WIKIPEDIA,
            "use_pyterrier_bm25": USE_PYTERRIER_BM25,
            "top_k_chunks": TOP_K_CHUNKS,
            "use_direct_model_first": USE_DIRECT_MODEL_FIRST,
            "use_direct_logit_scoring": USE_DIRECT_LOGIT_SCORING,
            "direct_logit_confidence_margin": DIRECT_LOGIT_CONFIDENCE_MARGIN,
            "use_agentic_tool_router": USE_AGENTIC_TOOL_ROUTER,
            "direct_tool_skip_margin": DIRECT_TOOL_SKIP_MARGIN,
            "verify_direct_with_wikipedia": VERIFY_DIRECT_WITH_WIKIPEDIA,
            "direct_model_votes": DIRECT_MODEL_VOTES,
            "min_seconds_for_direct_model": MIN_SECONDS_FOR_DIRECT_MODEL,
            "min_seconds_to_verify_direct": MIN_SECONDS_TO_VERIFY_DIRECT,
            "evidence_override_margin": EVIDENCE_OVERRIDE_MARGIN,
            "evidence_confidence_margin": EVIDENCE_CONFIDENCE_MARGIN,
            "min_evidence_score_to_trust": MIN_EVIDENCE_SCORE_TO_TRUST,
            "min_direct_margin_for_weak_evidence": MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE,
            "exact_option_match_boost": EXACT_OPTION_MATCH_BOOST,
            "option_term_coverage_weight": OPTION_TERM_COVERAGE_WEIGHT,
            "generate_rag_draft": GENERATE_RAG_DRAFT,
            "question_time_buffer": QUESTION_TIME_BUFFER,
            "min_seconds_to_attempt": MIN_SECONDS_TO_ATTEMPT,
            "delay_submit_for_wiki_cooldown": DELAY_SUBMIT_FOR_WIKI_COOLDOWN,
            "target_submit_elapsed_seconds": TARGET_SUBMIT_ELAPSED_SECONDS,
            "min_seconds_left_at_submit": MIN_SECONDS_LEFT_AT_SUBMIT,
            "max_submit_wait_seconds": MAX_SUBMIT_WAIT_SECONDS,
            "game_model": GAME_MODEL_ID,
            "submit_answers": SUBMIT_ANSWERS,
        },
        "questions": [],
    }

    print(f"Started game session {game.session_id}. Competition {COMPETITION_ID}.")
    question_count = 0
    correct_count = 0

    while game.in_progress:
        question = game.current_question
        if question is None:
            print("No active question returned by server.")
            break

        # Do not reset the Wikipedia cooldown here; the server rate limit continues across questions.

        question_count += 1
        current_level = game.current_level
        time_left = seconds_available(game)

        print("\n" + "=" * 80)
        print(f"Question {question_count} | Level {current_level} | {time_left:.1f}s left")
        print(question.text)
        for index, opt in enumerate(question.options):
            print(f"  {LETTERS[index]}. [{option_id(opt)}] {option_text(opt)}")
        print("=" * 80)

        if time_left < MIN_SECONDS_TO_ATTEMPT:
            selected = fallback_option(question)
            prediction = {
                "question": question.text,
                "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
                "documents": [],
                "rag_answer": "Skipped RAG because not enough time remained.",
                "rag_method": "fallback_time_guard",
                "evidence_chunks": [],
                "option_match": {
                    "answer_id": option_id(selected),
                    "answer_text": option_text(selected),
                    "answer_index": 0,
                    "letter": "A",
                    "model_output": "fallback_time_guard",
                },
                "elapsed_seconds": 0.0,
                "seconds_left_start": time_left,
                "seconds_left_end": time_left,
            }
        else:
            prediction = answer_one_question_with_pipeline(question, game=game)

        selected_id = prediction["option_match"]["answer_id"]
        selected_text = prediction["option_match"]["answer_text"]
        selected_letter = prediction["option_match"]["letter"]

        if VERBOSE:
            print("\nRAG answer:")
            print(prediction["rag_answer"])
            print("\nClosest option:", f"{selected_letter}. [{selected_id}] {selected_text}")
            print("Matcher output:", prediction["option_match"].get("model_output"))
            timings = prediction.get("timings", {})
            if timings:
                print(
                    "Timing:",
                    f"direct={timings.get('direct_model_seconds', 0):.2f}s",
                    f"wiki={timings.get('wikipedia_seconds', 0):.2f}s",
                    f"rag={timings.get('rag_generation_seconds', 0):.2f}s",
                    f"match={timings.get('option_matching_seconds', 0):.2f}s",
                    f"total={timings.get('pre_submit_pipeline_seconds', prediction['elapsed_seconds']):.2f}s",
                )
            print(f"Elapsed: {prediction['elapsed_seconds']:.2f}s | Time left now: {seconds_available(game):.1f}s")

        result_payload = None
        if SUBMIT_ANSWERS:
            submission_wait_seconds = wait_before_submit_for_cooldown(game)
            prediction["submission_wait_seconds"] = submission_wait_seconds
            if submission_wait_seconds:
                print(f"Time left after cooldown wait: {seconds_available(game):.1f}s")

            if seconds_available(game) <= QUESTION_TIME_BUFFER:
                print("Warning: low time before submit; submitting selected option immediately.")
            result = game.answer(selected_id)
            result_payload = {
                "correct": result.correct,
                "timed_out": result.timed_out,
                "game_over": result.game_over,
                "earned_amount": result.earned_amount,
            }
            correct_count += int(bool(result.correct))

            if result.correct:
                print(f"Correct. Earned: {result.earned_amount}")
            elif result.timed_out:
                print(f"Timed out. Earned: {result.earned_amount}")
            else:
                print(f"Wrong. Earned: {result.earned_amount}")
        else:
            print("Dry run: answer not submitted.")

        run_log["questions"].append(
            {
                "number": question_count,
                "level": current_level,
                "prediction": prediction,
                "result": result_payload,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )

        if result_payload and result_payload.get("game_over"):
            break
        if MAX_QUESTIONS is not None and question_count >= MAX_QUESTIONS:
            print("MAX_QUESTIONS reached; stopping.")
            break
        if not SUBMIT_ANSWERS:
            break

    run_log["finished_at"] = datetime.now(timezone.utc).isoformat()
    run_log["questions_answered"] = question_count
    run_log["correct_count"] = correct_count
    run_log["final_earned_amount"] = game.earned_amount

    if SAVE_RUN_LOG:
        log_dir = Path(RUN_LOG_DIR)
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f"test3_rag_game_{game.session_id}.json"
        with open(log_path, "w", encoding="utf-8") as handle:
            json.dump(run_log, handle, indent=2, ensure_ascii=False)
        print("Run log saved to:", log_path)

    print("\nGame summary")
    print("Questions answered:", question_count)
    print("Correct answers:", correct_count)
    print("Final earnings:", game.earned_amount)
    return game, run_log



In [ ]:
# =========================
# PRELOAD MODEL
# =========================
# Run this cell immediately before starting the actual timed game.

if PRELOAD_MODEL:
    preload_model_for_game()
else:
    print("PRELOAD_MODEL is False; the actual game will require an already loaded model.")


In [ ]:
# =========================
# ACTUAL GAME
# =========================
# Starts the timed game. Run the preload cell above first.

final_game, final_run_log = play_actual_rag_game()
